In [5]:
import pandas as pd
from pathlib import Path

In [ ]:
# Load data set
# cleaned data
data = pd.read_csv(r"..\..\..\Datasets\For analysis\full_data_CIFAR100_v3.csv")
# used to generate Results
#data = pd.read_csv(r"data.csv")

# Ensure we only have data with 80/20 datasplit
split_07 = [3911, 3912, 3913, 3914, 3915, 3916, 3917, 3918, 3919, 39110, 39111, 39112, 39113, 39114, 39115]
split_09 = [3901,3902,3903,3904,3905,3906,3907,3908,3909,39010,39011,39012,39013,39014,39015]
split = split_07 + split_09
data = data[~data["exp_id"].isin(split)]

# Rename names for readbility
rename_map = {
    "vit_l_32": "ViT-L/32",
    "vit_b_16": "ViT-B/16",
    "vit_b_32": "ViT-B/32",
    "vit_small_patch16_224": "ViT-S/16",
    "vit_small_patch32_224": "ViT-S/32",
    "vit_tiny_patch16_224": "ViT-T/16",
}

data["model"] = data["model"].replace(rename_map)


In [7]:
# Functions to analyse
def make_subset(df, model, bs, dropout, lr_to_assess):
# Freeze other hyperparameters so can get all configurations when changing batch rate while freezing other parameters

    subset = df[
        (df["model"] == model) &
        (df["batch_size"] == bs) &
        (df["lr"].isin(lr_to_assess)) &
        (df["dropout"] == dropout) & 
        (df["weight_decay"] == 0.0)
        
    ]
    return subset

def summary_table(subset):
    # Summary table that takes mean of every configuration
    view = (
        subset
        .groupby(["lr", "batch_size", "model", "dropout"])
        .agg(
            accuracy=("accuracy", "mean"),
            precision=("precision", "mean"),
            recall=("recall", "mean"),
            specificity=("specificity", "mean"),
            energy=("total_energy_J", "mean"),
            energy_std=("total_energy_J", "std"),
            n_seeds=("accuracy", "nunique"),
            Eg=("Eg","mean"),
            Pg=("Pg", "mean"),
            FPJ=("FPJ", "mean"),
            EDPinv=("EDPinv", "mean"),
            SPJ=("SPJ", "mean"),
        )
        .reset_index()
    )
    print("=========================================")
    print("View for plotting")
    print(view)
    print("=========================================")
    return view

def rank(results_df, metric, ascending = False):
 # Rank learning rates on a given metric, ie "accuracy" or "SPJ"
    df = results_df 
    return (
        df.groupby("lr")[metric]
        .mean()
        .sort_values(ascending=ascending)
        .reset_index()
        .rename(columns={metric: f"{metric}"})
    )


# Vit-b-32

In [8]:
lr_to_assess = [0.03 , 0.035, 0.04, 0.025, 0.02, 0.01]
vit_b_32 = make_subset(data,"ViT-B/32",128, 0.00, lr_to_assess)
vit_b_32 = summary_table(vit_b_32)


vit_b_32 = vit_b_32.dropna()
selected = vit_b_32
print("ViT-B/32")  
Eg = rank(selected,"Eg", ascending=False)
Pg = rank(selected, "Pg", ascending=False)
ac = rank(selected, "accuracy", ascending=False)
re = rank(selected, "recall", ascending=False)
sp = rank(selected, "specificity", ascending=False)
pre = rank(selected, "precision", ascending=False)
fpj = rank(selected, "FPJ", ascending=False)
edpinv = rank(selected, "EDPinv", ascending=False)
spj = rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = rank(selected,"EgPg", ascending=False)
energy = rank(selected,"energy", ascending=True)


print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))

print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))

print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg, Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

View for plotting
      lr  batch_size     model  dropout  accuracy  precision    recall  \
0  0.010         128  ViT-B/32      0.0  0.850500   0.850728  0.850278   
1  0.020         128  ViT-B/32      0.0  0.856490   0.856269  0.856421   
2  0.025         128  ViT-B/32      0.0  0.858145   0.857923  0.858076   
3  0.030         128  ViT-B/32      0.0  0.860072   0.859973  0.860095   
4  0.035         128  ViT-B/32      0.0  0.861080   0.860920  0.860970   
5  0.040         128  ViT-B/32      0.0  0.860740   0.860562  0.860655   

   specificity         energy    energy_std  n_seeds  Eg        Pg  \
0     0.998490  148786.007333   1055.304779        8 NaN  0.614298   
1     0.998550  226847.908340  87611.985769       19 NaN  0.627200   
2     0.998567  226697.951233  88181.291265       18 NaN  0.630861   
3     0.998587  235497.573596  89773.945547       17 NaN  0.635291   
4     0.998597  226717.852947  88501.827750       18 NaN  0.637393   
5     0.998593  226611.262778  88541.317306

# Vit-s-32

In [9]:
print("ViT-S/32")

lr_to_assess = [0.03 , 0.035, 0.04, 0.025, 0.02, 0.01]
vit_s_32 = make_subset(data,"ViT-S/32",128, 0.00, lr_to_assess)
vit_s_32 = summary_table(vit_s_32)
print("==========================================================")
print(vit_s_32)
print("==========================================================")

selected = vit_s_32

Eg = rank(selected,"Eg", ascending=False)
Pg = rank(selected, "Pg", ascending=False)
ac = rank(selected, "accuracy", ascending=False)
re = rank(selected, "recall", ascending=False)
sp = rank(selected, "specificity", ascending=False)
pre = rank(selected, "precision", ascending=False)
fpj = rank(selected, "FPJ", ascending=False)
edpinv = rank(selected, "EDPinv", ascending=False)
spj = rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = rank(selected,"EgPg", ascending=False)
energy = rank(selected,"energy", ascending=True)


print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg,  Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

ViT-S/32
View for plotting
      lr  batch_size     model  dropout  accuracy  precision    recall  \
0  0.010         128  ViT-S/32      0.0  0.875739   0.875741  0.875774   
1  0.020         128  ViT-S/32      0.0  0.880085   0.880028  0.880095   
2  0.025         128  ViT-S/32      0.0  0.880925   0.880866  0.880897   
3  0.030         128  ViT-S/32      0.0  0.879911   0.879959  0.879850   
4  0.035         128  ViT-S/32      0.0  0.879173   0.879098  0.879217   
5  0.040         128  ViT-S/32      0.0  0.877400   0.877493  0.877372   

   specificity        energy  energy_std  n_seeds  Eg        Pg  \
0     0.998745  76545.836969  394.415115       18 NaN  0.670834   
1     0.998789  77332.167872  491.566040       16 NaN  0.680829   
2     0.998797  77360.316521  430.473881       19 NaN  0.682745   
3     0.998787  77394.701430  555.526247       16 NaN  0.680438   
4     0.998780  77218.967182  593.876467       18 NaN  0.678721   
5     0.998762  77275.737793  496.208397       19 Na

# VIT-s-16


In [14]:
lr_to_assess = [0.03 , 0.035, 0.04, 0.025, 0.02, 0.01]
vit_s_16 = make_subset(data,"ViT-B/16",128, 0.00, lr_to_assess)
vit_s_16 = summary_table(vit_s_16)

selected = vit_s_16
print("ViT-S/16")
Eg = rank(selected,"Eg", ascending=False)
Pg = rank(selected, "Pg", ascending=False)
ac = rank(selected, "accuracy", ascending=False)
re = rank(selected, "recall", ascending=False)
sp = rank(selected, "specificity", ascending=False)
pre = rank(selected, "precision", ascending=False)
fpj = rank(selected, "FPJ", ascending=False)
edpinv = rank(selected, "EDPinv", ascending=False)
spj = rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = rank(selected,"EgPg", ascending=False)
energy = rank(selected,"energy", ascending=True)


print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg, Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

View for plotting
   batch_size     model    lr  dropout  accuracy  precision    recall  \
0         128  ViT-B/16  0.01      0.0  0.864600   0.864378  0.863532   
1         128  ViT-B/16  0.02      0.0  0.870200   0.870167  0.869682   
2         128  ViT-B/16  0.03      0.0  0.871829   0.872035  0.871784   
3         128  ViT-B/16  0.04      0.0  0.876800   0.876633  0.876313   

   specificity         energy   energy_std  n_seeds        Eg        Pg  \
0     0.998632  408302.517754          NaN        1  0.000610  0.644470   
1     0.998689  407767.930827   865.251213        2  0.000613  0.657691   
2     0.998705  405829.761279  2137.995943       11  0.000623  0.661951   
3     0.998756  409573.921708   124.568559        2  0.000602  0.672756   

             FPJ        EDPinv       SPJ  
0  422794.478539  1.228234e-09  1.175599  
1  423349.718463  1.230408e-09  1.177143  
2  425381.677695  1.239009e-09  1.182793  
3  421482.054679  1.219225e-09  1.171950  
ViT-S/16
PERFORMANCE


| 

# ViT Ti

In [ ]:

lr_to_assess = [0.03 , 0.035, 0.04, 0.025, 0.02, 0.01]
vit_t_16 = make_subset(data,"ViT-T/16",128, 0.00, lr_to_assess)
vit_t_16 = summary_table(vit_t_16)

selected = vit_t_16
print("ViT-Ti/16")
Eg = rank(selected,"Eg", ascending=False)
Pg = rank(selected, "Pg", ascending=False)
ac = rank(selected, "accuracy", ascending=False)
re = rank(selected, "recall", ascending=False)
sp = rank(selected, "specificity", ascending=False)
pre = rank(selected, "precision", ascending=False)
fpj = rank(selected, "FPJ", ascending=False)
edpinv = rank(selected, "EDPinv", ascending=False)
spj = rank(selected, "SPJ", ascending=False)
selected["EgPg"] = selected["Pg"] / selected["Eg"]
EgPg = rank(selected,"EgPg", ascending=False)
energy = rank(selected,"energy", ascending=True)


print("==========================================================================================")
print("PERFORMANCE")
print("==========================================================================================")
print("\n")
final = pd.concat([Pg, ac, re, sp, pre], axis=1)
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY")
print("==========================================================================================")
print("\n")
final = pd.concat([Eg, spj, edpinv, fpj, energy], axis=1).dropna()
print(final.to_markdown(index=False))


print("==========================================================================================")
print("ENERGY EFFICIENCY 2")
print("==========================================================================================")
print("\n")
final = pd.concat([EgPg, Eg, Pg, energy], axis=1).dropna()
print(final.to_markdown(index=False))

View for plotting
   batch_size     model     lr  dropout  accuracy  precision    recall  \
0         128  ViT-T/16  0.010      0.0  0.853467   0.853214  0.853635   
1         128  ViT-T/16  0.020      0.0  0.861545   0.861081  0.861605   
2         128  ViT-T/16  0.025      0.0  0.863564   0.863089  0.863596   
3         128  ViT-T/16  0.035      0.0  0.863545   0.863320  0.863673   
4         128  ViT-T/16  0.040      0.0  0.862791   0.862542  0.862898   

   specificity        energy   energy_std  n_seeds        Eg   Eg_norm  \
0     0.998520  92500.795673  2964.341669       18  0.010412  0.019927   
1     0.998601  91028.915227  2343.419745       11  0.010801  0.023946   
2     0.998622  91328.947420  2229.805871       11  0.010687  0.022937   
3     0.998622  90676.655718  1867.325950       11  0.010868  0.024815   
4     0.998614  90268.573575  1636.609065       11  0.010976  0.026000   

         Pg            FPJ        EDPinv       SPJ  FPJ_norm  EDPinv_norm  \
0  0.620710  12